# Semantic PDF Search API Testing

This notebook tests all the endpoints of the Semantic PDF Search API using the `requests` module.

**Make sure to start the API server first by running:**
```bash
python api.py
```

The server should be running at: http://localhost:8000

In [13]:
# Setup: Import required libraries and set base URL
import requests
import json
import os

# API base URL
BASE_URL = "http://localhost:8000"

print(f"API Base URL: {BASE_URL}")
print("Make sure the API server is running before executing the test cells!")

API Base URL: http://localhost:8000
Make sure the API server is running before executing the test cells!


## 1. Root Endpoint - GET /

Test the root endpoint to get API information and available endpoints.

In [14]:
# Test Root Endpoint - GET /
try:
    response = requests.get(f"{BASE_URL}/")
    
    print(f"Status Code: {response.status_code}")
    print(f"Response:")
    print(json.dumps(response.json(), indent=2))
    
except requests.exceptions.ConnectionError:
    print("❌ Connection Error: Make sure the API server is running!")
except Exception as e:
    print(f"❌ Error: {e}")

Status Code: 200
Response:
{
  "message": "Semantic PDF Search API",
  "version": "1.0.0",
  "endpoints": {
    "upload": "POST /upload - Upload PDF files",
    "search": "POST /search - Search through uploaded PDFs",
    "files": "GET /files - List uploaded files",
    "clear": "DELETE /clear - Clear all uploaded files"
  }
}


## 2. Upload PDFs - POST /upload

Upload PDF files to the server for indexing and semantic search.

In [23]:
# Debug: Test with minimal data
import requests

print("🔍 Debug: Testing upload endpoint with minimal data")
print("-" * 50)

try:
    # Create a simple text file to test upload mechanism
    test_content = b"PDF test content"
    files = {
        'files': ('test.pdf', test_content, 'application/pdf')
    }
    
    response = requests.post(f"{BASE_URL}/upload", files=files)
    
    print(f"Status Code: {response.status_code}")
    print(f"Response Text: '{response.text}'")
    print(f"Response Length: {len(response.text)}")
    
    if response.text.strip():
        try:
            print("Response JSON:")
            print(json.dumps(response.json(), indent=2))
        except json.JSONDecodeError as e:
            print(f"JSON Parse Error: {e}")
            print("Raw response content:")
            print(repr(response.content))
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

🔍 Debug: Testing upload endpoint with minimal data
--------------------------------------------------
Status Code: 200
Response Text: '{"message":"Successfully processed 1 PDF file(s)","files_processed":1,"files_list":[]}'
Response Length: 86
Response JSON:
{
  "message": "Successfully processed 1 PDF file(s)",
  "files_processed": 1,
  "files_list": []
}
Status Code: 200
Response Text: '{"message":"Successfully processed 1 PDF file(s)","files_processed":1,"files_list":[]}'
Response Length: 86
Response JSON:
{
  "message": "Successfully processed 1 PDF file(s)",
  "files_processed": 1,
  "files_list": []
}


In [32]:
# Test Upload PDFs - POST /upload (FIXED VERSION)
# Upload specific PDF files with better error handling

import os
import requests

# Define PDF file paths - using relative paths from your workspace
PDF_PATHS = [
    "Annual-Return-2023-24.pdf",
    "CDO_Kochi_Metro.pdf",
]

# Filter only existing files
existing_pdfs = []
workspace_path = r"c:\Users\Akhil\Projects\SIH_AI_Search"

for pdf_filename in PDF_PATHS:
    full_path = os.path.join(workspace_path, pdf_filename)
    if os.path.exists(full_path):
        existing_pdfs.append(full_path)
        print(f"✅ Found: {pdf_filename} ({os.path.getsize(full_path)} bytes)")
    else:
        print(f"⚠️  File not found: {pdf_filename}")

if existing_pdfs:
    try:
        print(f"\n📤 Preparing to upload {len(existing_pdfs)} files...")
        
        # Test with one file first to debug
        test_file_path = existing_pdfs[0]
        print(f"🧪 Testing with first file: {os.path.basename(test_file_path)}")
        
        with open(test_file_path, 'rb') as f:
            files = {
                'files': (os.path.basename(test_file_path), f, 'application/pdf')
            }
            
            print("📡 Making upload request...")
            response = requests.post(f"{BASE_URL}/upload", files=files)
            
            print(f"Status Code: {response.status_code}")
            print(f"Response Headers: {dict(response.headers)}")
            print(f"Response Text: {response.text}")
            
            if response.status_code == 200:
                print("✅ Upload successful!")
                try:
                    print("Response JSON:")
                    print(json.dumps(response.json(), indent=2))
                except:
                    print("Could not parse response as JSON")
            else:
                print("❌ Upload failed!")
                
    except Exception as e:
        print(f"❌ Error uploading files: {e}")
        import traceback
        print("Full traceback:")
        print(traceback.format_exc())
else:
    print("❌ No valid PDF files found!")

✅ Found: Annual-Return-2023-24.pdf (1094812 bytes)
✅ Found: CDO_Kochi_Metro.pdf (118303 bytes)

📤 Preparing to upload 2 files...
🧪 Testing with first file: Annual-Return-2023-24.pdf
📡 Making upload request...
Status Code: 200
Response Headers: {'date': 'Fri, 26 Sep 2025 16:24:55 GMT', 'server': 'uvicorn', 'content-length': '113', 'content-type': 'application/json'}
Response Text: {"message":"Successfully processed 1 PDF file(s)","files_processed":1,"files_list":["Annual-Return-2023-24.pdf"]}
✅ Upload successful!
Response JSON:
{
  "message": "Successfully processed 1 PDF file(s)",
  "files_processed": 1,
  "files_list": [
    "Annual-Return-2023-24.pdf"
  ]
}
Status Code: 200
Response Headers: {'date': 'Fri, 26 Sep 2025 16:24:55 GMT', 'server': 'uvicorn', 'content-length': '113', 'content-type': 'application/json'}
Response Text: {"message":"Successfully processed 1 PDF file(s)","files_processed":1,"files_list":["Annual-Return-2023-24.pdf"]}
✅ Upload successful!
Response JSON:
{
  "mes

## 3. Search PDFs - POST /search

Perform semantic search through the uploaded PDFs.

In [33]:
# Minimal Search Debug Test
print("🔧 Minimal Search Debug Test")
print("-" * 40)

try:
    # Test with the simplest possible search
    search_data = {"query": "test"}
    print(f"Testing with minimal query: '{search_data['query']}'")
    
    response = requests.post(
        f"{BASE_URL}/search",
        json=search_data,
        timeout=10
    )
    
    print(f"Status Code: {response.status_code}")
    print(f"Content Type: {response.headers.get('content-type', 'None')}")
    print(f"Response Length: {len(response.text)}")
    print(f"Response Text: '{response.text}'")
    
    if response.status_code != 200:
        print("❌ Server error - check your terminal for error logs!")
    else:
        print("✅ Search endpoint is working!")
        
except Exception as e:
    print(f"❌ Request failed: {e}")

🔧 Minimal Search Debug Test
----------------------------------------
Testing with minimal query: 'test'
Status Code: 500
Content Type: text/plain; charset=utf-8
Response Length: 21
Response Text: 'Internal Server Error'
❌ Server error - check your terminal for error logs!
Status Code: 500
Content Type: text/plain; charset=utf-8
Response Length: 21
Response Text: 'Internal Server Error'
❌ Server error - check your terminal for error logs!


In [29]:
# Simple search test with better error handling
print("🔍 Simple Search Test")
print("-" * 30)

try:
    # First, ensure we have files
    files_response = requests.get(f"{BASE_URL}/files")
    if files_response.status_code == 200:
        files_data = files_response.json()
        print(f"Available files: {files_data['total_files']}")
        
        if files_data['total_files'] == 0:
            print("❌ No files uploaded. Please upload files first!")
        else:
            # Try a very simple search
            search_data = {"query": "report"}
            print(f"Testing search with query: '{search_data['query']}'")
            
            search_response = requests.post(
                f"{BASE_URL}/search",
                headers={"Content-Type": "application/json"},
                json=search_data,
                timeout=30  # Longer timeout
            )
            
            print(f"Search Status: {search_response.status_code}")
            print(f"Search Headers: {dict(search_response.headers)}")
            print(f"Search Response: '{search_response.text}'")
            
            if search_response.status_code == 200:
                result = search_response.json()
                print("✅ Search successful!")
                print(f"Results: {len(result.get('results', []))}")
            else:
                print("❌ Search failed!")
    else:
        print("❌ Cannot check files endpoint")
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

🔍 Simple Search Test
------------------------------
Available files: 1
Testing search with query: 'report'
Available files: 1
Testing search with query: 'report'
Search Status: 500
Search Headers: {'date': 'Fri, 26 Sep 2025 16:19:50 GMT', 'server': 'uvicorn', 'content-length': '21', 'content-type': 'text/plain; charset=utf-8'}
Search Response: 'Internal Server Error'
❌ Search failed!
Search Status: 500
Search Headers: {'date': 'Fri, 26 Sep 2025 16:19:50 GMT', 'server': 'uvicorn', 'content-length': '21', 'content-type': 'text/plain; charset=utf-8'}
Search Response: 'Internal Server Error'
❌ Search failed!


## 4. List Files - GET /files

Get the list of all uploaded PDF files.

In [ ]:
# Test List Files - GET /files
try:
    response = requests.get(f"{BASE_URL}/files")
    
    print(f"Status Code: {response.status_code}")
    print("Response:")
    
    if response.status_code == 200:
        result = response.json()
        print(f"Total Files: {result['total_files']}")
        print("Uploaded Files:")
        for i, filename in enumerate(result['uploaded_files'], 1):
            print(f"  {i}. {filename}")
    else:
        print(json.dumps(response.json(), indent=2))
        
except Exception as e:
    print(f"❌ Error: {e}")

## 5. Clear Files - DELETE /clear

Clear all uploaded files and embeddings from the server.

In [22]:
# Test Clear Files - DELETE /clear
# WARNING: This will remove all uploaded files from the server!

print("⚠️  WARNING: This will clear all uploaded files from the server!")
confirmation = input("Are you sure you want to proceed? (yes/no): ")

if confirmation.lower() == 'yes':
    try:
        response = requests.delete(f"{BASE_URL}/clear")
        
        print(f"Status Code: {response.status_code}")
        print("Response:")
        print(json.dumps(response.json(), indent=2))
        
        # Verify files are cleared by checking the files endpoint
        print("\n✅ Verifying files are cleared...")
        verify_response = requests.get(f"{BASE_URL}/files")
        if verify_response.status_code == 200:
            result = verify_response.json()
            print(f"Files remaining: {result['total_files']}")
            
    except Exception as e:
        print(f"❌ Error: {e}")
else:
    print("❌ Operation cancelled.")

⚠️  WARNING: This will clear all uploaded files from the server!
Status Code: 200
Response:
{
  "message": "All files cleared successfully"
}

✅ Verifying files are cleared...
Status Code: 200
Response:
{
  "message": "All files cleared successfully"
}

✅ Verifying files are cleared...
Files remaining: 0
Files remaining: 0


## Additional: Complete API Test Flow

Run a complete test flow that demonstrates all API functionality in sequence.

In [ ]:
# Complete API Test Flow
def test_complete_flow():
    """Test the complete API workflow"""
    print("🚀 Starting Complete API Test Flow")
    print("=" * 50)
    
    # 1. Check API status
    print("\n1️⃣ Checking API Status...")
    try:
        response = requests.get(f"{BASE_URL}/")
        if response.status_code == 200:
            print("✅ API is running!")
        else:
            print("❌ API not responding correctly")
            return
    except:
        print("❌ Cannot connect to API. Make sure it's running!")
        return
    
    # 2. Clear any existing files
    print("\n2️⃣ Clearing existing files...")
    requests.delete(f"{BASE_URL}/clear")
    print("✅ Files cleared")
    
    # 3. Upload PDFs
    print("\n3️⃣ Uploading PDFs...")
    pdf_files = []
    workspace_path = r"c:\Users\Akhil\Projects\SIH_AI_Search"
    
    for file in os.listdir(workspace_path):
        if file.endswith('.pdf'):
            pdf_files.append(os.path.join(workspace_path, file))
    
    if pdf_files:
        files = []
        for pdf_path in pdf_files:
            with open(pdf_path, 'rb') as f:
                files.append(('files', (os.path.basename(pdf_path), f.read(), 'application/pdf')))
        
        response = requests.post(f"{BASE_URL}/upload", files=files)
        if response.status_code == 200:
            result = response.json()
            print(f"✅ Uploaded {result['files_processed']} files")
        else:
            print("❌ Upload failed")
            return
    else:
        print("❌ No PDF files found to upload")
        return
    
    # 4. List files
    print("\n4️⃣ Listing uploaded files...")
    response = requests.get(f"{BASE_URL}/files")
    result = response.json()
    print(f"✅ Found {result['total_files']} files: {result['uploaded_files']}")
    
    # 5. Test search
    print("\n5️⃣ Testing search...")
    test_query = "annual report"
    search_data = {"query": test_query}
    response = requests.post(f"{BASE_URL}/search", json=search_data)
    
    if response.status_code == 200:
        result = response.json()
        print(f"✅ Search successful for '{test_query}'")
        print(f"   Found {len(result['relevant_results'])} relevant results")
        if result['results']:
            top_result = result['results'][0]
            print(f"   Top match: {top_result['file']} ({top_result['match']}%)")
    else:
        print("❌ Search failed")
    
    print("\n🎉 Complete API Test Flow Finished!")
    print("=" * 50)

# Run the complete test
test_complete_flow()